In [1]:
import pymc as pm
import pytensor
import pytensor.tensor as pt
import jax
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist

ModuleNotFoundError: No module named 'pymc'

In [2]:
def extract_hrv_features(serie, window_size=20, window_size_long=40):
    """
    Toma una serie de intervalos RR y los tamaños de ventana para construir
    un DataFrame de características (features) y la variable objetivo (target).
    """
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)

    # 1. Generar las ventanas sobre la serie COMPLETA usando la ventana LARGA
    ventanas_long = sliding_window_view(serie, window_size_long)

    # 2. Separar las variables predictoras del target
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]

    # 3. Extraer la ventana CORTA (últimos 'window_size' valores)
    X_ventanas_short = X_ventanas_long[:, -window_size:]

    # 4. Calcular las diferencias dentro de cada ventana CORTA
    diffs = np.diff(X_ventanas_short, axis=1)

    # ---- Porta Index ----
    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)

    # Evitar división por cero en ventanas sin variabilidad
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    # ---- Guzik Index ----
    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    # Máscaras booleanas de diferencias
    mask_50 = np.abs(diffs) > 50
    mask_20 = np.abs(diffs) > 20

    # 5. Calcular features sobre las diferencias (ventana CORTA)
    nn50 = np.sum(mask_50, axis=1)
    nn20 = np.sum(mask_20, axis=1)
    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2) # variabilidad a corto plazo (Poincaré SD1)

    # 6. Calcular features sobre los valores absolutos (ventana CORTA)
    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    # 7. Calcular features sobre la ventana LARGA
    std_long = np.std(X_ventanas_long, axis=1)
    # Evitar raíces negativas limitando el valor mínimo a 0
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))  # variabilidad a largo plazo (Poincaré SD2)
    c_n = np.pi * sd1 * sd2  # área del elipse de Poincaré

    # 8. Calcular CCM sobre la ventana CORTA
    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i = ventanas_4puntos[:, 0]
    rr_i1 = ventanas_4puntos[:, 1]
    rr_i2 = ventanas_4puntos[:, 2]
    rr_i3 = ventanas_4puntos[:, 3]

    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))

    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    # 9. Crear el DataFrame combinando todo

    # NUEVO: Generar dinámicamente un diccionario con cada RR de la ventana corta
    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}

    # Diccionario con el resto de tus features estadísticas
    stats_columns = {
        'n_above': n_above,
        'n_below': n_below,
        'nn20': nn20,
        'nn50': nn50,
        'sdsd': sdsd,
        'mean': mean_val,
        'std': std_val,
        'var': var_val,
        'std_long': std_long,
        'sd1': sd1,
        'sd2': sd2,
        'c_n': c_n,
        'ccm': ccm,
        'porta': porta_index,
        'guzik': guzic_index,
        'target': y_target
    }

    # Combinamos ambos diccionarios para crear el DataFrame final
    # El operador ** desempaqueta los diccionarios
    df = pd.DataFrame({**rr_columns, **stats_columns})

    return df

In [3]:
def extract_hrv_features_2(serie, window_size=20, window_size_long=40):
    """
    Toma una serie de intervalos RR y los tamaños de ventana para construir
    un DataFrame de características ortogonales y la variable objetivo (target).
    """
    if window_size_long < window_size:
        raise ValueError("window_size_long debe ser mayor o igual a window_size")

    serie = np.asarray(serie, dtype=float)

    # 1. Generar ventanas sobre la serie COMPLETA
    ventanas_long = sliding_window_view(serie, window_size_long)

    # 2. Separar variables predictoras del target
    X_ventanas_long = ventanas_long[:-1]
    y_target = serie[window_size_long:]

    # 3. Extraer ventana CORTA
    X_ventanas_short = X_ventanas_long[:, -window_size:]

    # 4. Calcular diferencias dentro de la ventana CORTA
    diffs = np.diff(X_ventanas_short, axis=1)

    # ---- MÉTRICAS TRADICIONALES ----
    n_above = np.sum(diffs > 0, axis=1)
    n_below = np.sum(diffs < 0, axis=1)
    suma_porta = n_above + n_below
    porta_index = np.divide(n_below, suma_porta, out=np.zeros_like(n_below, dtype=float), where=suma_porta!=0)

    d_above = np.sum(np.abs(diffs) * (diffs > 0), axis=1) / np.sqrt(2)
    d_total = np.sum(np.abs(diffs), axis=1) / np.sqrt(2)
    guzic_index = np.divide(d_above, d_total, out=np.zeros_like(d_above, dtype=float), where=d_total!=0)

    nn50 = np.sum(np.abs(diffs) > 50, axis=1)
    nn20 = np.sum(np.abs(diffs) > 20, axis=1)

    sdsd = np.std(diffs, axis=1)
    sd1 = np.sqrt((sdsd**2) / 2)

    mean_val = np.mean(X_ventanas_short, axis=1)
    std_val = np.std(X_ventanas_short, axis=1)
    var_val = std_val ** 2

    std_long = np.std(X_ventanas_long, axis=1)
    inner_value = 2 * std_long**2 - sd1**2
    sd2 = np.sqrt(np.maximum(inner_value, 0))
    c_n = np.pi * sd1 * sd2

    # CCM
    ventanas_4puntos = sliding_window_view(X_ventanas_short, window_shape=4, axis=1)
    rr_i, rr_i1, rr_i2, rr_i3 = ventanas_4puntos[:, 0], ventanas_4puntos[:, 1], ventanas_4puntos[:, 2], ventanas_4puntos[:, 3]
    areas = 0.5 * np.abs(rr_i * (rr_i2 - rr_i3) - rr_i1 * (rr_i1 - rr_i3) + rr_i2 * (rr_i1 - rr_i2))
    denominador_ccm = c_n * (window_size - 2)
    ccm = np.divide(np.sum(areas, axis=1), denominador_ccm, out=np.zeros_like(c_n), where=denominador_ccm!=0)
    ccm = np.where(ccm > 1, 1, ccm)

    # -------------------------------------------------------------
    # NUEVAS CARACTERÍSTICAS ORTOGONALES (NO COLINEALES)
    # -------------------------------------------------------------

    # A. Coeficiente de Variación (CV)
    cv = np.divide(std_val, mean_val, out=np.zeros_like(std_val, dtype=float), where=mean_val!=0)

    # B. Robustez a Outliers: Rango Intercuartílico (IQR) y MAD
    q75, q25 = np.percentile(X_ventanas_short, [75, 25], axis=1)
    iqr = q75 - q25

    median_val = np.median(X_ventanas_short, axis=1)
    mad = np.median(np.abs(X_ventanas_short - median_val[:, None]), axis=1)

    # C. Fragmentación del Ritmo Cardíaco (PIP - Puntos de Inflexión)
    diffs_1 = diffs[:, :-1]
    diffs_2 = diffs[:, 1:]
    inflections = (diffs_1 * diffs_2) <= 0
    pip = np.sum(inflections, axis=1) / (window_size - 2)

    # D. Asimetría (Skewness) de las diferencias
    mean_diffs = np.mean(diffs, axis=1, keepdims=True)
    std_diffs = np.std(diffs, axis=1, keepdims=True)
    std_diffs_safe = np.where(std_diffs == 0, 1e-10, std_diffs) # Evitar NaN
    skewness = np.mean(((diffs - mean_diffs) / std_diffs_safe)**3, axis=1)

    # -------------------------------------------------------------
    # EMPAQUETADO FINAL
    # -------------------------------------------------------------
    rr_columns = {f'rr_{i+1}': X_ventanas_short[:, i] for i in range(window_size)}

    stats_columns = {
        'n_above': n_above, 'n_below': n_below, 'nn20': nn20, 'nn50': nn50,
        'sdsd': sdsd, 'mean': mean_val, 'std': std_val, 'var': var_val,
        'std_long': std_long, 'sd1': sd1, 'sd2': sd2, 'c_n': c_n,
        'ccm': ccm, 'porta': porta_index, 'guzik': guzic_index,
        'cv': cv, 'iqr': iqr, 'mad': mad, 'pip': pip, 'skewness': skewness, # <--- NUEVAS
        'target': y_target
    }

    df = pd.DataFrame({**rr_columns, **stats_columns})
    return df

In [4]:
# Cargar los datos
series_list = '/content/drive/MyDrive/Challenges_ML-DL/HRV_prediction/series/'
files = ['006.txt',
        '16265.txt',
        '16273.txt',
        '16786.txt',
        '16795.txt',
        '17453.txt',
        '18177.txt',
        '18184.txt',
        '16539.txt',
        'nsr054RRcl.txt',
        '005.txt',
        '16420.txt',
        '19140.txt',
        'nsr047RRcl.txt',
        'nsr048RRcl.txt',
        'nsr049RRcl.txt',
        '008.txt',
        '009.txt',
        '013.txt']
window_size = 20
np.random.seed(42)
SEED = 53
# select 50 random files
files_train = np.random.choice(files, size=11, replace=False) # np.array(['16795.txt', '013.txt', '18177.txt', 'nsr047RRcl.txt', '006.txt', '16786.txt', '19140.txt', 'nsr054RRcl.txt', '16420.txt', 'nsr048RRcl.txt'])

# the rest of the files will be used for testing
files_test = [f for f in files if f not in files_train]

# Crear DataFrames para entrenamiento y prueba
df_train = pd.DataFrame()
for file in files_train:
    data = np.loadtxt(os.path.join(series_list, file), dtype=int)
    df_temp = extract_hrv_features_2(data, window_size, window_size_long=40)

    # NUEVO: Agregamos una columna con el nombre del archivo para usarla en la estratificación
    df_temp['file_id'] = file

    df_train = pd.concat([df_train, df_temp], ignore_index=True)

# 1. Separar features (X), target (y) y grupos (file_id)
# Es VITAL quitar 'file_id' de X para que el modelo no lo use como feature predictivo
X = df_train.drop(columns=['target', 'file_id'])
y = df_train['target']
groups = df_train['file_id'] # Nuestra variable de estratificación para el K-Fold training

In [5]:
# --- 1. DEFINICIÓN DE VARIABLES ---
feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50', 'porta', 'std']
rr_cols = [f'rr_{i}' for i in range(1, 21)]
rr_last_col = 'rr_20'

X_feats = df_train[feature_cols].values
X_rr_seq = df_train[rr_cols].values  # Matriz de Nx20
y_target_raw = df_train['target'].values
rr_last_raw = df_train[rr_last_col].values

# Calcular la Diferencia
y_diff = y_target_raw - rr_last_raw

# --- 2. ESCALADO Y RESHAPE ---
scaler_feats = StandardScaler()
X_feats_scaled = scaler_feats.fit_transform(X_feats).astype('float32')

# ========================================================
# CORRECCIÓN: Escalar la secuencia de los 20 latidos GLOBALMENTE
# ========================================================
# 1. Aplanamos la matriz Nx20 a una sola columna de tamaño (N*20, 1)
X_rr_seq_flat = X_rr_seq.reshape(-1, 1)

# 2. Ajustamos el scaler a todos los latidos como si fueran una sola población
scaler_seq = StandardScaler()
X_rr_seq_scaled_flat = scaler_seq.fit_transform(X_rr_seq_flat).astype('float32')

# 3. Devolvemos los datos a su forma original (N, 20)
X_rr_seq_scaled = X_rr_seq_scaled_flat.reshape(X_rr_seq.shape)
# ========================================================

# REDIMENSIONAR PARA LA CNN/LSTM -> (Muestras, 20 latidos, 1 feature/canal)
X_rr_seq_3d = X_rr_seq_scaled.reshape(-1, 20, 1)

# Escalamos el DELTA
y_scaler = StandardScaler()
y_diff_scaled = y_scaler.fit_transform(y_diff.reshape(-1, 1)).flatten().astype('float32')

In [6]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv1D, BatchNormalization, Bidirectional, LSTM, Concatenate, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber, MeanSquaredError, LogCosh, MeanAbsoluteError
from tensorflow.keras.initializers import GlorotUniform, Orthogonal
import random

In [33]:
# 1. Fijar semilla de Python
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# 2. Fijar semilla de Numpy
np.random.seed(SEED)

# 3. Fijar semilla de TensorFlow (Afecta operaciones internas y Dropouts)
tf.random.set_seed(SEED)

# 4. Forzar operaciones deterministas (opcional pero recomendado)
# Esto asegura que TF no use algoritmos paralelos en la GPU/CPU que alteren los decimales
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# ==========================================
# RAMA 1: Procesamiento de la Serie Temporal
# ==========================================
# Entrada de la secuencia (20 pasos de tiempo, 1 canal)
input_seq = Input(shape=(20, 1), name='Input_Secuencia_RR')

# Capa Convolucional (4 filtros, kernel 3, activación lineal)
x_seq = Conv1D(filters=4, kernel_size=3, activation='linear', padding='valid', name='Conv1D_RR')(input_seq)
# Batch Normalization posterior
x_seq = BatchNormalization(name='BatchNorm_Conv')(x_seq)
# Activación Tanh (que añadiste recientemente)
x_seq = Activation('relu', name='tanh_Conv')(x_seq)

# LSTM Bidireccional (10 neuronas)
# Escupirá un vector de tamaño 20 (10 neuronas * 2 direcciones)
x_seq = Bidirectional(LSTM(8), name='BiLSTM_RR')(x_seq)

# ==========================================
# RAMA 2: Features Estáticas
# ==========================================
# Entrada de las 6 features
input_feats = Input(shape=(len(feature_cols),), name='Input_Features')

# ==========================================
# FUSIÓN Y CAPAS DENSAS
# ==========================================
# Concatenar la salida del LSTM (20) con las features estáticas (6) -> Vector de 26
merged = Concatenate(name='Concatenacion')([x_seq, input_feats])

# Primera Densa (52) -> Lineal -> BatchNorm -> ReLU -> Dropout
x = Dense(24, activation='linear', name='Dense_52')(merged)
x = BatchNormalization(name='BatchNorm_52')(x)
x = Activation('relu', name='ReLU_52')(x)
x = Dropout(0.2, name='Dropout_1')(x)  # <-- Primer Dropout

# Segunda Densa (16) -> Lineal -> BatchNorm -> ReLU -> Dropout
x = Dense(16, activation='linear', name='Dense_16')(x)
x = BatchNormalization(name='BatchNorm_16')(x)
x = Activation('relu', name='ReLU_16')(x)
x = Dropout(0.2, name='Dropout_2')(x)  # <-- Segundo Dropout

# Neurona de Salida (Target = Delta RR)
output_layer = Dense(1, activation='linear', name='Output_Delta_RR')(x)

# Crear el modelo final
cnn_lstm_model = Model(inputs=[input_seq, input_feats], outputs=output_layer)

# Ver un resumen de la arquitectura
cnn_lstm_model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Secuencia_RR  │ (None, 20, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1D_RR (Conv1D)  │ (None, 18, 4)     │         16 │ Input_Secuencia_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm_Conv      │ (None, 18, 4)     │         16 │ Conv1D_RR[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tanh_Conv           │ (None, 18, 4)     │          0 │ BatchNorm_Conv[0… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BiLSTM_RR           │ (None, 16)        │        832 │ tanh_Conv[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Input_Features      │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Concatenacion       │ (None, 24)        │          0 │ BiLSTM_RR[0][0],  │
│ (Concatenate)       │                   │            │ Input_Features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_52 (Dense)    │ (None, 24)        │        600 │ Concatenacion[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm_52        │ (None, 24)        │         96 │ Dense_52[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ReLU_52             │ (None, 24)        │          0 │ BatchNorm_52[0][… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_1 (Dropout) │ (None, 24)        │          0 │ ReLU_52[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_16 (Dense)    │ (None, 16)        │        400 │ Dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ BatchNorm_16        │ (None, 16)        │         64 │ Dense_16[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ReLU_16             │ (None, 16)        │          0 │ BatchNorm_16[0][… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dropout_2 (Dropout) │ (None, 16)        │          0 │ ReLU_16[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Output_Delta_RR     │ (None, 1)         │         17 │ Dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,041 (7.97 KB)

 Trainable params: 1,953 (7.63 KB)

 Non-trainable params: 88 (352.00 B)

In [34]:
# Compilar
cnn_lstm_model.compile(
    optimizer=Adam(learning_rate=0.005),
    loss=Huber(delta=0.8),
    metrics=['mae', 'mse']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print(f"Iniciando entrenamiento Híbrido CNN-BiLSTM. Filas totales: {X_rr_seq_3d.shape[0]}...")

# Entrenar pasándole la lista de entradas: [Secuencia 3D, Features 2D]
history = cnn_lstm_model.fit(
    x=[X_rr_seq_3d, X_feats_scaled],
    y=y_diff_scaled,
    epochs=100,
    batch_size=2048,
    validation_split=0.15,
    callbacks=[early_stopping],
    verbose=1
)

Iniciando entrenamiento Híbrido CNN-BiLSTM. Filas totales: 1044950...
Epoch 1/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - loss: 0.2368 - mae: 0.5616 - mse: 0.7596 - val_loss: 0.3293 - val_mae: 0.6984 - val_mse: 1.1327
Epoch 2/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: 0.2151 - mae: 0.5276 - mse: 0.6702 - val_loss: 0.3124 - val_mae: 0.6719 - val_mse: 1.0732
Epoch 3/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.2105 - mae: 0.5203 - mse: 0.6513 - val_loss: 0.3096 - val_mae: 0.6676 - val_mse: 1.0639
Epoch 4/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - loss: 0.2085 - mae: 0.5170 - mse: 0.6422 - val_loss: 0.3130 - val_mae: 0.6728 - val_mse: 1.0724
Epoch 5/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.2074 - mae: 0.5153 - mse: 0.6380 - val_loss: 0.3093 - val_mae: 0.6670 - val_mse: 1.0608
Epoch 6/100
434/434 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - loss: 0.2067 - mae: 0.5142 - mse: 0.6355 - val_loss: 0.3089 - val_mae: 0.6665 - val_mse: 1.0582
Epoch 7/100
434/

In [35]:
# --- 1. PREPARACIÓN DE DATOS DE PRUEBA ---
df_test = pd.DataFrame()
for file in files_test:
    data = np.loadtxt(os.path.join(series_list, file), dtype=int)
    df_temp = extract_hrv_features_2(data, window_size, window_size_long=40)
    df_temp['file_id'] = file
    df_test = pd.concat([df_test, df_temp], ignore_index=True)

# [Extracción asumiendo df_test ya preparado]
X_feats_test = df_test[feature_cols].values
X_rr_seq_test = df_test[rr_cols].values  # Matriz de Nx20
y_test_raw = df_test['target'].values
rr_last_test = df_test[rr_last_col].values

# --- 2. ESCALADO (CON LA CORRECCIÓN 3D) ---
X_feats_test_scaled = scaler_feats.transform(X_feats_test).astype('float32')

# ========================================================
# CORRECCIÓN PARA TESTEO: Aplanar, Escalar, Reconstruir
# ========================================================
# 1. Aplanamos la matriz Nx20 a una sola columna (N*20, 1)
X_rr_seq_test_flat = X_rr_seq_test.reshape(-1, 1)

# 2. Transformamos la columna plana usando el scaler_seq ajustado
X_rr_seq_test_scaled_flat = scaler_seq.transform(X_rr_seq_test_flat).astype('float32')

# 3. Devolvemos los datos a su forma original (N, 20)
X_rr_seq_test_scaled = X_rr_seq_test_scaled_flat.reshape(X_rr_seq_test.shape)
# ========================================================

# Reshape a 3D vital para Keras (N, 20, 1)
X_rr_seq_test_3d = X_rr_seq_test_scaled.reshape(-1, 20, 1)

# --- 3. PREDICCIÓN PASANDO LA LISTA DE ENTRADAS ---
print(f"Iniciando predicciones. Filas totales: {X_rr_seq_test_3d.shape[0]}...")

y_pred_diff_scaled = cnn_lstm_model.predict([X_rr_seq_test_3d, X_feats_test_scaled], batch_size=50000)

# --- 4. DESESCALAR Y RECONSTRUIR ---
y_pred_diff_ms = y_scaler.inverse_transform(y_pred_diff_scaled).flatten()
y_pred_ms = y_pred_diff_ms + rr_last_test

print("¡Predicciones terminadas exitosamente!")

# --- 5. EVALUACIÓN DE MÉTRICAS GLOBALES ---
mae = mean_absolute_error(y_test_raw, y_pred_ms)
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_ms))
r2 = r2_score(y_test_raw, y_pred_ms)

print(f"\n--- Métricas en Test de No Vistos (CNN-LSTM Hybrid) ---")
print(f"MAE:  {mae:.2f} ms")
print(f"RMSE: {rmse:.2f} ms")
print(f"R²:   {r2:.4f}")

# --- 6. EVALUACIÓN DEL OBJETIVO DE NEGOCIO ---
errores_absolutos = np.abs(y_test_raw - y_pred_ms)
latidos_en_tolerancia = np.mean(errores_absolutos <= 15.0)

print(f"Porcentaje de intervalos RR predichos dentro de la tolerancia (±15ms): {latidos_en_tolerancia:.2%}")

Iniciando predicciones. Filas totales: 804505...
17/17 ━━━━━━━━━━━━━━━━━━━━ 13s 746ms/step
¡Predicciones terminadas exitosamente!

--- Métricas en Test de No Vistos (CNN-LSTM Hybrid) ---
MAE:  20.68 ms
RMSE: 31.64 ms
R²:   0.9513
Porcentaje de intervalos RR predichos dentro de la tolerancia (±15ms): 52.10%


In [37]:
import json
import os

# Define la ruta donde guardarás tus archivos (puedes cambiarla)
save_dir = '/content/drive/MyDrive/Challenges_ML-DL/HRV_prediction/'
os.makedirs(save_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. GUARDAR EL MODELO KERAS (H5 o Keras format)
# ---------------------------------------------------------
# Guardar solo los pesos es más ligero, pero guardar todo el modelo (.keras)
# incluye la arquitectura, lo que evita tener que reescribir el código para instanciarlo luego.
model_path = os.path.join(save_dir, 'cnn_lstm_hybrid.keras')
cnn_lstm_model.save(model_path)
print(f"✅ Modelo híbrido guardado en: {model_path}")

# ---------------------------------------------------------
# 2. GUARDAR LOS SCALERS EN FORMATO JSON
# ---------------------------------------------------------
# StandardScaler almacena la media en `.mean_` y la varianza en `.var_`.
# Extraemos esos arrays de Numpy, los convertimos a listas de Python
# (porque JSON no entiende de Numpy) y los empaquetamos.

scalers_dict = {
    # 1. Scaler de la Secuencia RR (20 latidos)
    "scaler_seq": {
        "mean": scaler_seq.mean_.tolist(),
        "var": scaler_seq.var_.tolist(),
        "scale": scaler_seq.scale_.tolist()  # scale_ es la desviación estándar (raíz de la varianza)
    },

    # 2. Scaler de las Features Estáticas (8 features)
    "scaler_feats": {
        "mean": scaler_feats.mean_.tolist(),
        "var": scaler_feats.var_.tolist(),
        "scale": scaler_feats.scale_.tolist()
    },

    # 3. Scaler de la variable Y (Delta RR)
    "y_scaler": {
        "mean": y_scaler.mean_.tolist(),
        "var": y_scaler.var_.tolist(),
        "scale": y_scaler.scale_.tolist()
    }
}

# Escribir el diccionario a un archivo JSON
json_path = os.path.join(save_dir, 'scalers_params.json')
with open(json_path, 'w') as json_file:
    json.dump(scalers_dict, json_file, indent=4)

print(f"✅ Parámetros de escalado guardados en: {json_path}")

✅ Modelo híbrido guardado en: /content/drive/MyDrive/Challenges_ML-DL/HRV_prediction/cnn_lstm_hybrid.keras
✅ Parámetros de escalado guardados en: /content/drive/MyDrive/Challenges_ML-DL/HRV_prediction/scalers_params.json
